# 02 — Preprocessing

Cleans, resamples, and chronologically partitions each raw water-level series.

**Inputs:** `data/raw/*.parquet`
**Outputs:** separate station artifacts in `data/processed/separate/` plus joined `data/processed/joined/all_stations_train.parquet` and `all_stations_test.parquet`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
## Setup

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    COUNTRY_CODE,
    MAX_INTERPOLATION_GAP_HOURS,
    STATION_IDS,
    TARGET_STATION_ID,
    TEST_FRACTION,
    WEATHER_VARIABLES,
)
from src.fetch_data import summarize_failures
from src.preprocess import (
    join_station_frames,
    preprocess_station,
    split_train_test,
    write_preprocess_artifacts,
)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
SEPARATE_DIR = PROCESSED_DIR / "separate"
JOINED_DIR = PROCESSED_DIR / "joined"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Per-station hourly water level + weather

Each station's water-level history is reindexed to a strict hourly UTC grid;
gaps of at most `MAX_INTERPOLATION_GAP_HOURS` are linearly interpolated and
flagged via `imputed`, longer gaps stay `NaN`. GeoSphere INCA weather is left-joined
onto that grid, since the water-level timeline is the forecasting target. The first
`floor((1 - TEST_FRACTION) * N)` rows are written as training data and the
remainder as a physically sealed test artifact.

In [ ]:
latest_station = None
train_by_station = {}
test_by_station = {}
failures = {}

for station_id in tqdm(STATION_IDS, desc="Preprocessing", unit="station"):
    try:
        station = preprocess_station(
            station_id,
            raw_dir=RAW_DIR,
            max_gap_hours=MAX_INTERPOLATION_GAP_HOURS,
            weather_variables=WEATHER_VARIABLES,
        )
        train, test = split_train_test(station, TEST_FRACTION)
        manifest = write_preprocess_artifacts(
            train,
            test,
            station_id=station_id,
            output_dir=SEPARATE_DIR,
            test_fraction=TEST_FRACTION,
        )
        train_by_station[station_id] = train
        test_by_station[station_id] = test
        print(
            f"Saved {len(train):,} train and {len(test):,} test rows for {station_id}"
        )
        latest_station = train
    except Exception as error:  # noqa: BLE001 -- aggregate every station failure
        failures[station_id] = error
        print(f"Failed {station_id}: {error}")

if latest_station is not None:
    latest_station.info()
    display(latest_station.head())
else:
    print("No preprocessed DataFrame is available to display.")

In [ ]:
if failures:
    raise RuntimeError(summarize_failures(failures))

print("Preprocessing completed successfully.")

## Joined target and upstream station frames

The target station supplies the timestamp timeline. Every non-timestamp column is prefixed with its station identifier, and upstream values are missing where a station has no observation for a target timestamp. Train and test partitions are joined independently to preserve the chronological split.

In [ ]:
station_catalog = pd.read_parquet(
    RAW_DIR / f"pegelalarm_stations_{COUNTRY_CODE.lower()}.parquet"
)
station_positions = station_catalog.set_index("commonid")["positionKm"]
target_position_km = float(station_positions.loc[TARGET_STATION_ID])

def distance_to_target(station_id: str) -> float:
    return abs(float(station_positions.loc[station_id]) - target_position_km)

station_order = [
    TARGET_STATION_ID,
    *sorted(
        (station_id for station_id in STATION_IDS if station_id != TARGET_STATION_ID),
        key=distance_to_target,
    ),
]
station_distances = {
    station_id: distance_to_target(station_id) for station_id in station_order
}
print(
    "Join order (nearest upstream station first): "
    + " -> ".join(
        f"{station_id} ({station_distances[station_id]:.1f} km)"
        for station_id in station_order
    )
)

ordered_train_by_station = {
    station_id: train_by_station[station_id] for station_id in station_order
}
ordered_test_by_station = {
    station_id: test_by_station[station_id] for station_id in station_order
}

joined_train = join_station_frames(
    ordered_train_by_station, target_station_id=TARGET_STATION_ID
)
joined_test = join_station_frames(
    ordered_test_by_station, target_station_id=TARGET_STATION_ID
)

JOINED_DIR.mkdir(parents=True, exist_ok=True)
joined_train_path = JOINED_DIR / "all_stations_train.parquet"
joined_test_path = JOINED_DIR / "all_stations_test.parquet"
joined_train.to_parquet(joined_train_path, index=False)
joined_test.to_parquet(joined_test_path, index=False)

print(f"Saved joined train dataset: {joined_train_path} ({len(joined_train):,} rows)")
print(f"Saved joined test dataset: {joined_test_path} ({len(joined_test):,} rows)")
joined_train.info()
display(joined_train)